<a href="https://colab.research.google.com/github/sameer-04062004/A-Multimodal-approach-for-Detecting-Depression/blob/main/CNN__Video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Connecting Drive to Google Colab**

In [1]:
from google.colab import drive
drive.mount('/content/drive')
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
from keras import metrics
from sklearn.metrics import classification_report
from keras.models import Sequential
from keras.layers import Conv1D
from keras.layers import MaxPooling1D
from keras.layers import Flatten
from keras.layers import Dense

Mounted at /content/drive


In [2]:
def processData(data):
    X = data.iloc[:,:].values
    X = np.delete(X, 0, 1)
    X = np.delete(X, 1, 1)
    for i in range(len(X)):
        if(isinstance(X[i][5],str) or isinstance(X[i][7],str)):
            X[i] = np.zeros((1, X.shape[1]))
            # print("se" , end = " ")
    return X

def getData(filename):
    data = pd.read_csv(filename,delimiter=',', engine='python')
    X = processData(data)
    return X

def makeDataPoint(patientNo):
    file1 = (patientNo)+"_CLNF_AUs.txt"
    file2 = (patientNo)+"_CLNF_features.txt"
    file3 = (patientNo)+"_CLNF_features3D.txt"
    file4 = (patientNo)+"_CLNF_gaze.txt"
    file5 = (patientNo)+"_CLNF_hog.txt"
    file6 = (patientNo)+"_CLNF_pose.txt"

    X1 = getData(file1)
    X2 = getData(file2)
    X3 = getData(file3)
    X4 = getData(file4)
    X6 = getData(file6)

    X = np.concatenate((X1, X2, X3, X4, X6), 1)
    return X

In [11]:
def scale_down(X):
  X_new = []
  size = 2
  for i in range(int(X.shape[0]/size)):
    cur_row = X[i*size]
    for j in range(1,size):
      if(i+j < X.shape[0]):
        cur_row += X[i+j]
    cur_row = cur_row/size
    X_new.append(cur_row)
  X_new = np.array(X_new)
  return X_new

def decrease_size(X):
  size = 10000
  if(X.shape[0] < size):
    dif = size - X.shape[0]
    temp = np.zeros((dif,X.shape[1]))
    X = np.concatenate((X,temp),axis = 0)
  elif(X.shape[0] > size):
    X = X[:10000, :]
  return X

def makeDataset(location, folder):
    file  = np.array(pd.read_csv(location,delimiter=',',encoding='utf-8'))[:, 0:2]
    X_temp = []
    Y_temp = []
    for i in range(len(file)):
      patientID = (str(int(file[i][0])))
      string = '/content/drive/MyDrive/DAIC/' + folder +"/"+ patientID
      XT = makeDataPoint(string)
      XT = scale_down(XT)
      XT = decrease_size(XT)
      X_temp.append(XT)
      Y_temp.append(int(file[i][1]))
    Y_temp = np.asarray(Y_temp)
    return X_temp, Y_temp

In [12]:
class CNN_Video:
  def __init__(self):
    # Initialising the CNN
    classifier = Sequential()
    # Step 1 - Convolution
    classifier.add(Conv1D(300, 3, input_shape = (10000, 388), activation = 'relu'))
    # Step 2 - Pooling
    classifier.add(MaxPooling1D(pool_size = 2))
    classifier.add(Conv1D(150, 3, activation = 'relu'))
    classifier.add(MaxPooling1D(pool_size = 2))
    # Adding a second convolutional layer
    classifier.add(Conv1D(75, 3, activation = 'relu'))
    classifier.add(MaxPooling1D(pool_size = 2))
    classifier.add(Conv1D(32, 3, activation = 'relu'))
    classifier.add(MaxPooling1D(pool_size = 2))
    # Step 3 - Flattening
    classifier.add(Flatten())
    # Step 4 - Full connection
    classifier.add(Dense(units = 128, activation = 'relu'))
    classifier.add(Dense(units = 1, activation = 'sigmoid'))
  # Compiling the CNN
    classifier.compile(optimizer = 'adam', loss = 'binary_crossentropy')
    class_weight = { 0: 0.3, 1:0.7}
    self.classifier = classifier

  def fitModel(self, XtrainTotal, YtrainTotal, epoch = 5):
    return self.classifier.fit(XtrainTotal, YtrainTotal, epochs=epoch)

  def predictModel(self, X_test):
    return self.classifier.predict(np.asarray(X_test))

In [13]:
import os

print(os.listdir('/content/drive/MyDrive/DAIC'))

['341_P', '337_P', '338_P', '336_P', '345_P', '346_P', '335_P', '339_P', '334_P', '344_P', '340_P', '347_P', '348_P', '343_P', '349_P', '350_P', '351_P', '300_P', '354_P', '303_P', '304_P', '302_P', '353_P', '301_P', '306_P', '352_P', '305_P', '307_P', '308_P', '310_P', '311_P', '309_P', '320_P', '318_P', '314_P', '315_P', '313_P', '317_P', '321_P', '316_P', '312_P', '322_P', '326_P', '324_P', '319_P', '323_P', '325_P', '327_P', '332_P', '330_P', '329_P', '333_P', '328_P', '331_P', '355_P', '356_P', '400_P', '361_P', '363_P', '358_P', '357_P', '366_P', '360_P', '359_P', '364_P', '367_P', '369_P', '368_P', '365_P', '362_P', '372_P', '371_P', '375_P', '370_P', '374_P', '380_P', '376_P', '381_P', '373_P', '382_P', '378_P', '377_P', '379_P', '383_P', '385_P', '384_P', '386_P', '395_P', '389_P', '388_P', '399_P', '393_P', '387_P', '390_P', '396_P', '391_P', '397_P', '392_P', 'dev_split_Depression_AVEC2017.csv', 'full_test_split.csv', 'test_split_Depression_AVEC2017.csv', 'train_split_Depres

In [ ]:
# # Get Dataset
X_train, Y_train = makeDataset('/content/drive/MyDrive/DAIC/train_split_Depression_AVEC2017.csv', 'train_data')
X_dev, Y_dev = makeDataset('/content/drive/MyDrive/DAIC/dev_split_Depression_AVEC2017.csv', 'dev_data')
X_test, Y_test = makeDataset('/content/drive/MyDrive/DAIC/full_test_split.csv', 'test_data')
YtrainTotal = np.concatenate((Y_train, Y_dev))
XtrainTotal = X_train + X_dev
XtrainTotal = np.asarray(XtrainTotal, dtype=np.float32)

# # Run Model On Test Set
model = CNN_Video()
model.fitModel(XtrainTotal, YtrainTotal, 5)
Y_pred = model.predictModel(X_test)
print(classification_report(Y_test,Y_pred))
